# Customer Churn Survival Analysis — Walkthrough

This notebook walks through a complete survival-analysis workflow using simulated subscriber data.

1. Generate a synthetic cohort
2. Fit Kaplan-Meier survival curves
3. Run log-rank tests
4. Find peak empirical hazard windows
5. Fit a Cox proportional hazards model
6. Score 90-day churn risk


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import matplotlib.pyplot as plt

from synth_subscribers import generate_subscribers
from kaplan_meier import fit_overall, fit_by_group, summary_by_group, logrank, peak_hazard_window
from cox_model import prepare_features, fit_cox, hazard_ratios, risk_score_at_horizon


## 1. Generate the synthetic cohort


In [ ]:
df = generate_subscribers(n=50_000, seed=2024)
print(f'Subscribers: {len(df):,}')
print(f'Churn rate: {df.observed.mean():.1%}')
df.head()


## 2. Overall Kaplan-Meier curve


In [ ]:
kmf = fit_overall(df)
ax = kmf.plot_survival_function(figsize=(10, 5), ci_show=True)
ax.set_title('Overall subscriber survival curve')
ax.set_xlabel('Days since signup')
ax.set_ylabel('S(t) = P(active)')
ax.grid(True, alpha=0.3)
print(f'Median survival time: {kmf.median_survival_time_:.0f} days')


## 3. Survival by billing dispute


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, curve in fit_by_group(df, 'had_billing_dispute').items():
    curve.plot_survival_function(ax=ax, label=f'dispute={label}')
ax.set_title('Survival by early billing dispute status')
ax.set_xlabel('Days since signup')
ax.set_ylabel('S(t)')
ax.grid(True, alpha=0.3)

lr = logrank(df, 'had_billing_dispute')
print(f'Log-rank test: chi2={lr.test_statistic:.2f}, p={lr.p_value:.4g}')
summary_by_group(df, 'had_billing_dispute')


## 4. Peak hazard window


In [ ]:
haz = peak_hazard_window(df, bin_days=15, max_day=365).dropna()
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(haz['day_start'], haz['hazard'], width=12, alpha=0.7)
ax.set_title('Empirical hazard rate by 15-day window')
ax.set_xlabel('Days since signup')
ax.set_ylabel('Hazard rate')
peak = haz.loc[haz['hazard'].idxmax()]
print(f'Peak hazard window: days {peak.day_start}-{peak.day_end} (rate {peak.hazard:.4f})')


## 5. Cox proportional hazards model


In [ ]:
features = prepare_features(df)
cph = fit_cox(features)
hazard_ratios(cph).head(15)


## 6. 90-day churn risk scoring


In [ ]:
risks_90 = risk_score_at_horizon(cph, features, days=90)
df_scored = df.copy()
df_scored['churn_risk_90d'] = risks_90.values
df_scored.nlargest(15, 'churn_risk_90d')[[
    'subscriber_id', 'plan_tier', 'payment_method',
    'had_billing_dispute', 'support_tickets_30d', 'churn_risk_90d'
]]
